# Lossless MDL graph compression for molecular graphs

This is the cleaned, package-backed chemistry entry point. It does **not**
redefine the compressor and does not bundle QM9. It demonstrates:

1. loading or constructing labeled molecular graphs;
2. fitting a graphlet dictionary on training graphs only;
3. transforming held-out graphs;
4. verifying exact labeled decoding; and
5. exposing ordinary NetworkX graphs for downstream models.

For QM9, first run `python scripts/prepare_qm9.py` and install the chemistry
extra with `python -m pip install -e ".[chem]"`.

In [ ]:
from pathlib import Path
import os
import math

import networkx as nx
import pandas as pd

from buhito.mdl import MDLGraphCompressor, labeled_isomorphic

## Dataset adapter

Set `BUHITO_QM9_CSV` to a CSV containing a lowercase `smiles` column. When the
variable is absent, the notebook uses a small deterministic molecular-like
fixture so every cell remains runnable without external data.

In [ ]:
def synthetic_molecular_graphs():
    graphs = []
    for repeats in range(3, 11):
        fragments = []
        for index in range(repeats):
            fragment = nx.cycle_graph(3) if index % 2 == 0 else nx.path_graph(4)
            nx.set_node_attributes(fragment, "C", "atom_key")
            nx.set_edge_attributes(fragment, "SINGLE", "bond_key")
            fragments.append(fragment)
        graphs.append(nx.disjoint_union_all(fragments))
    return graphs


def load_graphs(limit=200):
    csv_value = os.environ.get("BUHITO_QM9_CSV")
    if not csv_value:
        print("BUHITO_QM9_CSV is not set; using the synthetic fixture.")
        return synthetic_molecular_graphs()

    from buhito.converters import smiles_to_nx

    csv_path = Path(csv_value).expanduser()
    frame = pd.read_csv(csv_path, nrows=limit)
    if "smiles" not in frame:
        raise ValueError(f"{csv_path} must contain a lowercase 'smiles' column")

    graphs = []
    for smiles in frame["smiles"].astype(str):
        graph, _ = smiles_to_nx(smiles)
        graphs.append(graph)
    return graphs


graphs = load_graphs()
len(graphs), graphs[0].number_of_nodes(), graphs[0].number_of_edges()

In [ ]:
split = max(2, int(0.75 * len(graphs)))
train_graphs = graphs[:split]
test_graphs = graphs[split:] or graphs[-2:]

compressor = MDLGraphCompressor(
    graphlet_sizes=(3,),
    n_rules=5,
    min_graph_support=2,
    min_occurrences=4,
    max_candidates=50,
    node_label_keys="atom_key",
    edge_label_keys="bond_key",
    selector="sparse",
    dictionary_selection="best",
    cache_dir="artifacts/notebook_cache/chemistry",
    validate=True,
    progress=True,
)
compressor.fit(train_graphs)
result = compressor.transform(test_graphs)

In [ ]:
display(compressor.dictionary_frame())
display(result.per_graph)
result.report

In [ ]:
decoded = result.decoded_graphs()
assert all(
    labeled_isomorphic(original, reconstructed)
    for original, reconstructed in zip(test_graphs, decoded)
)
print(f"Exact labeled decoding verified for {len(decoded)} graphs.")

compressed_for_models = result.model_graphs()
[(g.number_of_nodes(), g.number_of_edges()) for g in compressed_for_models[:5]]

## Interpretation

A negative `net_savings_bits` value is still a valid result: the dictionary and
boundary metadata cost more than the graphlet contractions save. The exact
round-trip assertion above is independent of whether compression is beneficial.